# ReCAHS — Multi-seed Static and Joint Regime-Aware Pruning

Bu notebook her model seed'i için maskeleri **yeniden seçer** ve üç yöntemi karşılaştırır: unpruned baseline, global static %25 pruning ve joint/greedy regime-aware %25 pruning.

Bilimsel güvenlik kontrolleri:
- Time-Series-Library commit'i sabittir.
- Validation/test loader'ları `shuffle=False` kurulur; STL label satırları pencere sırasıyla eşleşir.
- Her checkpoint'in baseline test MSE'si eğitim sonucuyla karşılaştırılır.
- Static ve joint maskeler her seed için validation verisinden ayrı seçilir.
- Test verisi yalnızca final değerlendirmede kullanılır.
- Pencere bazlı hatalar block-bootstrap analizi için saklanır.

> Nihai deneyler `SEEDS = [7, 42, 1234, 2026, 3407]` ile çalıştırılır; mevcut seed çıktıları bulunduğunda yeniden kullanılabilir.

In [ ]:
# 1) AYARLAR
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/BIL401_Regime_Head_Pruning')
TSLIB_DIR = Path('/content/Time-Series-Library')
TSLIB_COMMIT = '4e938a1767106324dd753b2a44832bf870a0252e'
DATA_URL = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv'

# Nihai beş-seed deney listesi.
SEEDS = [7, 42, 1234, 2026, 3407]
GREEDY_SUBSAMPLE_SIZE = 256
GREEDY_SELECTION_SEED = 42
KEEP_HEADS = 18       # 24 head'in %75'i aktif
CONFIDENCE_THRESHOLD = 0.05
FORCE_RERUN = False

MULTISEED_DIR = PROJECT_DIR / 'multiseed' / 'ETTh1'
REGIME_DIR = PROJECT_DIR / 'regime_detection'
OUTPUT_DIR = MULTISEED_DIR / 'pruning'

print('Seeds:', SEEDS)
print('Output:', OUTPUT_DIR)

In [ ]:
# 2) DRIVE, TSLIB, PAKETLER VE DATASET
from google.colab import drive

# İlk hücre Drive bağlanmadan /content/drive altında klasör oluşturmaz.
# Mount noktası önceki bir denemeden doluysa güvenli bir alternatif kullanılır.
primary_mount = Path('/content/drive')
if (primary_mount / 'MyDrive').is_dir():
    drive_root = primary_mount
    print('Drive zaten bağlı:', drive_root)
else:
    try:
        drive.mount(str(primary_mount))
        drive_root = primary_mount
    except ValueError as error:
        if 'already contain files' not in str(error):
            raise
        drive_root = Path('/content/gdrive')
        if not (drive_root / 'MyDrive').is_dir():
            drive.mount(str(drive_root))
        print('Dolu eski mount noktası atlandı; Drive buraya bağlandı:', drive_root)

PROJECT_DIR = drive_root / 'MyDrive' / 'BIL401_Regime_Head_Pruning'
MULTISEED_DIR = PROJECT_DIR / 'multiseed' / 'ETTh1'
REGIME_DIR = PROJECT_DIR / 'regime_detection'
OUTPUT_DIR = MULTISEED_DIR / 'pruning'
for directory in [MULTISEED_DIR, REGIME_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import urllib.request
from argparse import Namespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

def run(command, cwd=None, check=True):
    print('$', ' '.join(map(str, command)))
    result = subprocess.run(list(map(str, command)), cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Komut başarısız: {command}')
    return result

if not (TSLIB_DIR / '.git').exists():
    run(['git', 'clone', 'https://github.com/thuml/Time-Series-Library.git', TSLIB_DIR])
run(['git', 'fetch', 'origin'], cwd=TSLIB_DIR)
run(['git', 'checkout', '--force', TSLIB_COMMIT], cwd=TSLIB_DIR)
assert run(['git', 'rev-parse', 'HEAD'], cwd=TSLIB_DIR).stdout.strip() == TSLIB_COMMIT

# Yalnızca bu notebook'un doğrudan kullandığı eksik bağımlılıklar; mevcut NumPy/PyTorch ortamı değiştirilmez.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'patool', 'sktime', 'scikit-base', 'einops',
    'reformer-pytorch', 'local-attention', 'hyper-connections',
    'axial-positional-embedding', 'product-key-memory', 'colt5-attention',
])
import reformer_pytorch

drive_data = PROJECT_DIR / 'data' / 'ETTh1.csv'
drive_data.parent.mkdir(parents=True, exist_ok=True)
if not drive_data.exists():
    urllib.request.urlretrieve(DATA_URL, drive_data)
raw_df = pd.read_csv(drive_data)
assert raw_df.shape == (17420, 8), raw_df.shape
assert raw_df.columns.tolist() == ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
tslib_data = TSLIB_DIR / 'dataset' / 'ETDataset' / 'ETT-small' / 'ETTh1.csv'
tslib_data.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(drive_data, tslib_data)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

dataset_sha256 = sha256_file(drive_data)
if str(TSLIB_DIR) not in sys.path: sys.path.insert(0, str(TSLIB_DIR))
os.chdir(TSLIB_DIR)
print('Dataset SHA256:', dataset_sha256)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) VALIDATION VE TEST STL REGIME ETİKETLERİ
from statsmodels.tsa.seasonal import STL

SEQ_LEN, PRED_LEN, STL_PERIOD = 336, 96, 24
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 12*30*24, 4*30*24, 4*30*24

def label_window(series_window):
    fit = STL(series_window, period=STL_PERIOD, robust=True).fit()
    variances = np.array([np.var(fit.trend), np.var(fit.seasonal), np.var(fit.resid)], dtype=float)
    scores = variances / variances.sum() if variances.sum() > 1e-12 else np.zeros(3)
    names = ['trend', 'seasonal', 'residual']
    order = np.argsort(scores)[::-1]
    return {
        'regime': names[int(order[0])],
        'trend_score': float(scores[0]), 'seasonal_score': float(scores[1]), 'residual_score': float(scores[2]),
        'confidence_margin': float(scores[order[0]] - scores[order[1]]),
    }

def build_or_load_labels(split):
    path = REGIME_DIR / f'etth1_{split}_regimes_ot_seq336.csv'
    expected = 2785
    if path.exists():
        labels = pd.read_csv(path)
        if len(labels) == expected and set(labels['regime'].unique()) <= {'trend','seasonal','residual'}:
            if 'window_id' not in labels: labels.insert(0, 'window_id', np.arange(len(labels)))
            labels = labels.sort_values('window_id').reset_index(drop=True)
            if 'confidence_margin' not in labels:
                values = labels[['trend_score','seasonal_score','residual_score']].to_numpy()
                values.sort(axis=1)
                labels['confidence_margin'] = values[:, -1] - values[:, -2]
            labels['is_confident'] = labels['confidence_margin'] >= CONFIDENCE_THRESHOLD
            print(f'{split}: mevcut etiketler kullanıldı:', path)
            return labels

    border1 = TRAIN_SIZE - SEQ_LEN if split == 'validation' else TRAIN_SIZE + VAL_SIZE - SEQ_LEN
    border2 = TRAIN_SIZE + VAL_SIZE if split == 'validation' else TRAIN_SIZE + VAL_SIZE + TEST_SIZE
    values = raw_df.iloc[border1:border2]['OT'].to_numpy(dtype=np.float64)
    count = len(values) - SEQ_LEN - PRED_LEN + 1
    records = []
    for window_id in tqdm(range(count), desc=f'STL {split}'):
        records.append({'window_id': window_id, **label_window(values[window_id:window_id+SEQ_LEN])})
    labels = pd.DataFrame(records)
    labels['is_confident'] = labels['confidence_margin'] >= CONFIDENCE_THRESHOLD
    labels.to_csv(path, index=False)
    return labels

val_labels = build_or_load_labels('validation')
test_labels = build_or_load_labels('test')
assert len(val_labels) == len(test_labels) == 2785
display(pd.DataFrame({'validation': val_labels.regime.value_counts(), 'test': test_labels.regime.value_counts()}))

In [ ]:
# 4) MODEL, NON-SHUFFLED LOADERS VE HEAD MASK CONTROLLER
from torch.utils.data import DataLoader, Subset
from data_provider.data_loader import Dataset_ETT_hour
from models.PatchTST import Model as PatchTSTModel

args = Namespace(
    task_name='long_term_forecast', data='ETTh1', root_path=str(tslib_data.parent) + '/', data_path='ETTh1.csv',
    features='M', target='OT', freq='h', embed='timeF', seasonal_patterns='Monthly',
    seq_len=336, label_len=48, pred_len=96, enc_in=7, dec_in=7, c_out=7,
    d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, factor=3,
    dropout=0.1, activation='gelu', augmentation_ratio=0.0, batch_size=32, num_workers=0
)
timeenc = 1
val_data = Dataset_ETT_hour(args=args, root_path=args.root_path, flag='val', size=[336,48,96], features='M', data_path='ETTh1.csv', target='OT', timeenc=timeenc, freq='h')
test_data = Dataset_ETT_hour(args=args, root_path=args.root_path, flag='test', size=[336,48,96], features='M', data_path='ETTh1.csv', target='OT', timeenc=timeenc, freq='h')
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=0, drop_last=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0, drop_last=False)
assert len(val_data) == len(val_labels) and len(test_data) == len(test_labels)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

class HeadMaskController:
    def __init__(self, model):
        self.model, self.original_forwards, self.current_mask = model, {}, None
    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention
            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward
            def make_forward(layer_idx, attention_layer):
                def masked_forward(queries, keys, values, attn_mask, tau=None, delta=None):
                    B, L, _ = queries.shape; _, S, _ = keys.shape; H = attention_layer.n_heads
                    q = attention_layer.query_projection(queries).view(B,L,H,-1)
                    k = attention_layer.key_projection(keys).view(B,S,H,-1)
                    v = attention_layer.value_projection(values).view(B,S,H,-1)
                    out, attn = attention_layer.inner_attention(q,k,v,attn_mask,tau=tau,delta=delta)
                    if self.current_mask is not None:
                        mask = self.current_mask.to(out.device)
                        if mask.ndim == 2:
                            layer_mask = mask[layer_idx].view(1,1,H,1)
                        else:
                            if mask.shape[0] != B:
                                if B % mask.shape[0] != 0: raise ValueError((mask.shape, B))
                                mask = mask.repeat_interleave(B // mask.shape[0], dim=0)
                            layer_mask = mask[:,layer_idx,:].view(B,1,H,1)
                        out = out * layer_mask
                    return attention_layer.out_projection(out.reshape(B,L,-1)), attn
                return masked_forward
            attention_layer.forward = make_forward(layer_idx, attention_layer)
    def set_mask(self, mask): self.current_mask = mask.detach().clone().float()

mse_none = nn.MSELoss(reduction='none')
mae_none = nn.L1Loss(reduction='none')
print('Validation/test loaders shuffle=False olarak hazır:', len(val_data), len(test_data))

In [ ]:
# 5) DEĞERLENDİRME VE MASK SEÇİM FONKSİYONLARI
def evaluate(model, loader, labels, controller, global_mask=None, regime_masks=None, desc='eval'):
    model.eval(); records=[]; offset=0
    with torch.no_grad():
        for batch_x, batch_y, batch_x_mark, batch_y_mark in tqdm(loader, desc=desc, leave=False):
            batch_x=batch_x.float().to(device); batch_y=batch_y.float().to(device)
            batch_x_mark=batch_x_mark.float().to(device); batch_y_mark=batch_y_mark.float().to(device)
            batch_size=batch_x.shape[0]
            if regime_masks is None:
                controller.set_mask(global_mask)
            else:
                batch_masks=[regime_masks[labels.iloc[offset+i].regime] for i in range(batch_size)]
                controller.set_mask(torch.stack(batch_masks))
            output=model(batch_x,batch_x_mark,batch_y,batch_y_mark); true=batch_y[:,-args.pred_len:,:]
            mse=mse_none(output,true).mean(dim=(1,2)); mae=mae_none(output,true).mean(dim=(1,2))
            for i in range(batch_size):
                row=labels.iloc[offset+i]
                records.append({'window_id':offset+i,'regime':row.regime,'is_confident':bool(row.is_confident),'mse':float(mse[i]),'mae':float(mae[i])})
            offset += batch_size
    frame=pd.DataFrame(records)
    return {'mse':float(frame.mse.mean()),'mae':float(frame.mae.mean()),'windows':frame}

def mask_from_active(active):
    mask=torch.zeros(3,8)
    for layer,head in active: mask[layer,head]=1
    return mask

def collect_regime_batches(regime):
    indices=val_labels.index[val_labels.regime == regime].tolist()
    rng=random.Random(GREEDY_SELECTION_SEED)
    if len(indices)>GREEDY_SUBSAMPLE_SIZE: indices=sorted(rng.sample(indices,GREEDY_SUBSAMPLE_SIZE))
    loader=DataLoader(Subset(val_data,indices),batch_size=32,shuffle=False,num_workers=0)
    batches=[]
    for x,y,xm,ym in loader: batches.append((x.float().to(device),y.float().to(device),xm.float().to(device),ym.float().to(device)))
    return batches, indices

def eval_batches(model, controller, mask, batches):
    controller.set_mask(mask); model.eval(); total=0.0; count=0
    with torch.no_grad():
        for x,y,xm,ym in batches:
            pred=model(x,xm,y,ym); true=y[:,-args.pred_len:,:]
            values=mse_none(pred,true).mean(dim=(1,2)); total += values.sum().item(); count += len(values)
    return total/count

def greedy_joint(model, controller, batches, regime):
    active=set((l,h) for l in range(3) for h in range(8)); rows=[]
    while len(active)>KEEP_HEADS:
        best_head=None; best_mse=None
        for candidate in sorted(active):
            trial=mask_from_active(active-{candidate}); score=eval_batches(model,controller,trial,batches)
            if best_mse is None or score < best_mse - 1e-12 or (abs(score-best_mse)<=1e-12 and candidate<best_head):
                best_head,best_mse=candidate,score
        active.remove(best_head)
        rows.append({'step':len(rows)+1,'layer':best_head[0],'head':best_head[1],'resulting_mse':best_mse,'remaining_heads':len(active)})
        print(f'  {regime} step {len(rows)}: remove {best_head}, mse={best_mse:.6f}')
    return mask_from_active(active), pd.DataFrame(rows)

In [ ]:
# 6) TEK SEED İÇİN TÜM DENEY
baseline_summary_path = MULTISEED_DIR / 'baseline_multiseed_summary.csv'
baseline_training_summary = pd.read_csv(baseline_summary_path)
all_heads_mask=torch.ones(3,8)
regime_caches={regime:collect_regime_batches(regime) for regime in ['trend','seasonal','residual']}

def find_checkpoint(seed):
    candidates=list((MULTISEED_DIR/f'seed_{seed}'/'checkpoints').glob('**/checkpoint.pth'))
    if len(candidates)!=1: raise RuntimeError(f'seed={seed}: {len(candidates)} checkpoint bulundu')
    return candidates[0]

def run_seed(seed):
    seed_out=OUTPUT_DIR/f'seed_{seed}'; seed_out.mkdir(parents=True,exist_ok=True)
    done_path=seed_out/'summary.csv'
    if done_path.exists() and not FORCE_RERUN:
        done=pd.read_csv(done_path)
        if set(done.method)=={'unpruned_baseline','static_25','dynamic_joint_25'}:
            print(f'seed={seed} tamamlanmış, atlandı.')
            return done

    checkpoint=find_checkpoint(seed); model=PatchTSTModel(args).to(device)
    state=torch.load(checkpoint,map_location=device,weights_only=True); model.load_state_dict(state); model.eval()
    controller=HeadMaskController(model); controller.install()

    print(f'\nSEED {seed}: baseline doğrulama')
    base_val=evaluate(model,val_loader,val_labels,controller,global_mask=all_heads_mask,desc=f'{seed} baseline val')
    base_test=evaluate(model,test_loader,test_labels,controller,global_mask=all_heads_mask,desc=f'{seed} baseline test')
    expected=float(baseline_training_summary.loc[baseline_training_summary.seed==seed,'test_mse'].iloc[0])
    if abs(base_test['mse']-expected)>1e-5:
        raise RuntimeError(f'Baseline uyuşmuyor: custom={base_test["mse"]}, training={expected}')

    print(f'SEED {seed}: global head importance ve static mask')
    importance=[]
    for layer in range(3):
        for head in range(8):
            mask=all_heads_mask.clone(); mask[layer,head]=0
            result=evaluate(model,val_loader,val_labels,controller,global_mask=mask,desc=f'{seed} L{layer}H{head}')
            importance.append({'layer':layer,'head':head,'masked_val_mse':result['mse'],'importance':result['mse']-base_val['mse']})
    importance_df=pd.DataFrame(importance).sort_values(['importance','layer','head']).reset_index(drop=True)
    importance_df.to_csv(seed_out/'global_head_importance.csv',index=False)
    pruned = [
    (int(r["layer"]), int(r["head"]))
    for _, r in importance_df.head(6).iterrows()
]
    static_mask=all_heads_mask.clone()
    for layer,head in pruned: static_mask[layer,head]=0
    pd.DataFrame(static_mask.numpy()).to_csv(seed_out/'static_keep_mask.csv',index=False)
    static_val=evaluate(model,val_loader,val_labels,controller,global_mask=static_mask,desc=f'{seed} static val')
    static_test=evaluate(model,test_loader,test_labels,controller,global_mask=static_mask,desc=f'{seed} static test')

    print(f'SEED {seed}: joint/greedy regime maskeleri')
    joint_masks={}; removal_frames=[]
    for regime,(batches,indices) in regime_caches.items():
        mask,removal=greedy_joint(model,controller,batches,regime); joint_masks[regime]=mask
        removal.insert(0,'regime',regime); removal_frames.append(removal)
        pd.DataFrame(mask.numpy()).to_csv(seed_out/f'{regime}_joint_keep_mask.csv',index=False)
    pd.concat(removal_frames,ignore_index=True).to_csv(seed_out/'joint_removal_orders.csv',index=False)
    joint_val=evaluate(model,val_loader,val_labels,controller,regime_masks=joint_masks,desc=f'{seed} joint val')
    joint_test=evaluate(model,test_loader,test_labels,controller,regime_masks=joint_masks,desc=f'{seed} joint test')

    windows=base_test['windows'][['window_id','regime','is_confident']].copy()
    windows['baseline_mse']=base_test['windows'].mse; windows['static_mse']=static_test['windows'].mse; windows['dynamic_joint_mse']=joint_test['windows'].mse
    windows['baseline_mae']=base_test['windows'].mae; windows['static_mae']=static_test['windows'].mae; windows['dynamic_joint_mae']=joint_test['windows'].mae
    windows.to_csv(seed_out/'test_window_losses.csv',index=False)

    rows=[]
    for method,val_result,test_result in [('unpruned_baseline',base_val,base_test),('static_25',static_val,static_test),('dynamic_joint_25',joint_val,joint_test)]:
        rows.append({'dataset':'ETTh1','seed':seed,'method':method,'val_mse':val_result['mse'],'val_mae':val_result['mae'],'test_mse':test_result['mse'],'test_mae':test_result['mae'],'checkpoint_sha256':sha256_file(checkpoint),'dataset_sha256':dataset_sha256,'tslib_commit':TSLIB_COMMIT})
    summary=pd.DataFrame(rows); summary.to_csv(done_path,index=False)
    del model,controller; torch.cuda.empty_cache()
    return summary

In [ ]:
# 7) ÇALIŞTIR, BİRLEŞTİR VE MEAN ± STD RAPORLA
seed_summaries=[]
for seed in SEEDS: seed_summaries.append(run_seed(seed))

all_summary_files=sorted(OUTPUT_DIR.glob('seed_*/summary.csv'))
combined=pd.concat([pd.read_csv(path) for path in all_summary_files],ignore_index=True)
combined=combined.drop_duplicates(['dataset','seed','method'],keep='last').sort_values(['seed','method']).reset_index(drop=True)
combined_path=OUTPUT_DIR/'pruning_multiseed_summary.csv'; combined.to_csv(combined_path,index=False)
display(combined[['seed','method','val_mse','test_mse','test_mae']])

stats=(combined.groupby('method').agg(test_mse_mean=('test_mse','mean'),test_mse_std=('test_mse','std'),test_mae_mean=('test_mae','mean'),test_mae_std=('test_mae','std'),seeds=('seed','nunique')).reset_index())
stats.to_csv(OUTPUT_DIR/'pruning_mean_std.csv',index=False)
display(stats)
print('\nÖzet:', combined_path)
print('Pencere hataları:', OUTPUT_DIR/'seed_XXXX'/'test_window_losses.csv')
